# Run 7 — Ablation skip connections sur données complètes

**Objectif** : Vérifier si le résultat no_skip > with_skip (observé sur 50% des données) se confirme sur 100%.

On entraîne deux modèles avec la config du modèle final, l'un avec skip, l'autre sans,
puis on évalue les deux sur le test set officiel ISBI 2016 (379 images).

In [6]:
import os, time, gc, json as _json
import glob as _glob
import numpy as np

# --- GPU config (Onyxia / cuDNN stability) ---
os.environ['TF_XLA_FLAGS'] = '--tf_xla_auto_jit=0'
os.environ['TF_CUDNN_USE_AUTOTUNE'] = '0'
os.environ['TF_USE_CUDNN_BATCHNORM_SPATIAL_PERSISTENT'] = '0'

import tensorflow as tf
tf.config.optimizer.set_experimental_options({'disable_meta_optimizer': True})

from tensorflow.keras import layers, models, backend as K
from sklearn.model_selection import StratifiedShuffleSplit
from pathlib import Path
from PIL import Image
import pandas as pd

print(f"TensorFlow {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

2026-03-14 19:55:39.551518: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-14 19:55:39.620858: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-14 19:55:40.751473: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [7]:
# ── Configuration ──
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

_kaggle = _glob.glob('/kaggle/input/*/dataset_ISIC')
_base = _kaggle[0] if _kaggle else 'dataset_ISIC'

IMAGES_DIR = os.path.join(_base, 'ISBI2016_ISIC_Part1_Training_Data')
MASKS_DIR  = os.path.join(_base, 'ISBI2016_ISIC_Part1_Training_GroundTruth')
TEST_IMG_DIR  = os.path.join(_base, 'ISBI2016_ISIC_Part1_Test_Data')
TEST_MASK_DIR = os.path.join(_base, 'ISBI2016_ISIC_Part1_Test_GroundTruth')

SAVE_DIR = 'models'
os.makedirs(SAVE_DIR, exist_ok=True)

IMG_SIZE = 256
EPOCHS = 30
PATIENCE = 8
LR = 1e-4
BATCH_SIZE = 8
AUTOTUNE = tf.data.AUTOTUNE

print(f"Images: {IMAGES_DIR}")
print(f"Test: {TEST_IMG_DIR}")

Images: dataset_ISIC/ISBI2016_ISIC_Part1_Training_Data
Test: dataset_ISIC/ISBI2016_ISIC_Part1_Test_Data


## 1. Fonctions utilitaires

In [8]:
# ── Data loading ──
def load_image_mask(img_path, mask_path):
    img  = tf.image.decode_jpeg(tf.io.read_file(img_path), channels=3)
    mask = tf.image.decode_png(tf.io.read_file(mask_path), channels=1)
    img  = tf.image.convert_image_dtype(img, tf.float32)
    mask = tf.cast(mask > 127, tf.float32)
    return img, mask

def preprocess(img, mask, img_size=(IMG_SIZE, IMG_SIZE)):
    img  = tf.image.resize(img, img_size, method='bilinear')
    mask = tf.image.resize(mask, img_size, method='nearest')
    return img, mask

def augment_flip(img, mask):
    if tf.random.uniform(()) > 0.5:
        img  = tf.image.flip_left_right(img)
        mask = tf.image.flip_left_right(mask)
    if tf.random.uniform(()) > 0.5:
        img  = tf.image.flip_up_down(img)
        mask = tf.image.flip_up_down(mask)
    return img, mask

def make_ds(img_files, mask_files, batch_size=BATCH_SIZE, augment_fn=None, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((img_files, mask_files))
    ds = ds.map(lambda i, m: load_image_mask(i, m), num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda i, m: preprocess(i, m), num_parallel_calls=AUTOTUNE)
    if augment_fn:
        ds = ds.map(augment_fn, num_parallel_calls=AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(len(img_files))
    ds = ds.batch(batch_size).prefetch(AUTOTUNE)
    return ds

# ── Metrics & Loss ──
def dice_coefficient(y_true, y_pred, smooth=1e-6):
    y_true_f, y_pred_f = K.flatten(y_true), K.flatten(y_pred)
    inter = K.sum(y_true_f * y_pred_f)
    return (2. * inter + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def iou_coefficient(y_true, y_pred, smooth=1e-6):
    y_true_f, y_pred_f = K.flatten(y_true), K.flatten(y_pred)
    inter = K.sum(y_true_f * y_pred_f)
    union = K.sum(y_true_f) + K.sum(y_pred_f) - inter
    return (inter + smooth) / (union + smooth)

def dice_loss(y_true, y_pred):
    return 1.0 - dice_coefficient(y_true, y_pred)

def bce_dice_loss(y_true, y_pred):
    return tf.keras.losses.binary_crossentropy(y_true, y_pred) + dice_loss(y_true, y_pred)

METRICS = [dice_coefficient, iou_coefficient, 'binary_accuracy']
CUSTOM_OBJECTS = {
    'dice_coefficient': dice_coefficient,
    'iou_coefficient': iou_coefficient,
    'dice_loss': dice_loss,
    'bce_dice_loss': bce_dice_loss,
}

# ── Model ──
# IMPORTANT : config identique au FINAL original
# batchnorm=True, pooling='max' (vérifié dans hyperparam_unet_v2.ipynb)
def conv_block(x, filters, activation='relu', use_batchnorm=True):
    for _ in range(2):
        x = layers.Conv2D(filters, 3, padding='same', kernel_initializer='he_normal')(x)
        if use_batchnorm:
            x = layers.BatchNormalization()(x)
        x = layers.Activation(activation)(x)
    return x

def build_unet(input_shape=(IMG_SIZE, IMG_SIZE, 3), base_filters=64, depth=4,
               use_skip=True, dropout_rate=0.0, activation='relu',
               upsample='bilinear', use_batchnorm=True, pooling='max'):
    inputs = layers.Input(shape=input_shape)
    skips = []
    x = inputs
    pool_layer = layers.MaxPool2D if pooling == 'max' else layers.AveragePooling2D
    for i in range(depth):
        x = conv_block(x, base_filters * (2**i), activation, use_batchnorm)
        skips.append(x)
        x = pool_layer((2,2))(x)
    x = conv_block(x, base_filters * (2**depth), activation, use_batchnorm)
    if dropout_rate > 0:
        x = layers.Dropout(dropout_rate)(x)
    for i in reversed(range(depth)):
        filters_i = base_filters * (2**i)
        if upsample == 'transpose':
            x = layers.Conv2DTranspose(filters_i, 2, strides=2, padding='same',
                                       kernel_initializer='he_normal')(x)
        else:
            x = layers.UpSampling2D((2,2))(x)
        if use_skip:
            x = layers.Concatenate()([x, skips[i]])
        x = conv_block(x, filters_i, activation, use_batchnorm)
    outputs = layers.Conv2D(1, 1, activation='sigmoid')(x)
    return models.Model(inputs, outputs)

# ── Cache helpers ──
def _safe_name(name):
    return name.replace(' ', '_').replace('/', '-').replace('(', '').replace(')', '')

def _save_result(name, res):
    path = os.path.join(SAVE_DIR, f'{_safe_name(name)}.json')
    with open(path, 'w') as f:
        _json.dump(res, f)
    print(f'  Resultat sauvegarde -> {path}')

def _load_result(name):
    path = os.path.join(SAVE_DIR, f'{_safe_name(name)}.json')
    if os.path.exists(path):
        with open(path) as f:
            return _json.load(f)
    return None

results = []

def run_experiment(name, model, train_ds, val_ds, lr=LR, loss_fn=dice_loss):
    cached = _load_result(name)
    if cached is not None:
        print(f'\n{"="*60}')
        print(f'  {name}  |  CACHE')
        print(f'  -> Dice={cached["best_val_dice"]:.4f}  IoU={cached["best_val_iou"]:.4f}')
        print(f'{"="*60}')
        if not any(r['name'] == name for r in results):
            results.append(cached)
        del model; K.clear_session(); gc.collect()
        return cached

    model.compile(optimizer=tf.keras.optimizers.Adam(lr), loss=loss_fn, metrics=METRICS)
    cb = [tf.keras.callbacks.EarlyStopping(patience=PATIENCE, monitor='val_loss',
                                           restore_best_weights=True)]
    print(f'\n{"="*60}')
    print(f'  {name}  |  params: {model.count_params():,}')
    print(f'{"="*60}')
    t0 = time.time()
    hist = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=cb, verbose=2)
    elapsed = time.time() - t0

    history_dict = {k: [float(x) for x in v] for k, v in hist.history.items()}
    best_dice = max(history_dict['val_dice_coefficient'])
    best_iou  = max(history_dict['val_iou_coefficient'])

    res = dict(name=name, best_val_dice=best_dice, best_val_iou=best_iou,
               params=model.count_params(), time_s=round(elapsed, 1),
               history=history_dict, stopped_epoch=len(history_dict['loss']))

    if not any(r['name'] == name for r in results):
        results.append(res)

    print(f'  -> Dice={best_dice:.4f}  IoU={best_iou:.4f}  ({elapsed:.0f}s, {len(history_dict["loss"])} epochs)')

    model_path = os.path.join(SAVE_DIR, f'{_safe_name(name)}.keras')
    model.save(model_path)
    print(f'  Modele sauvegarde -> {model_path}')
    _save_result(name, res)

    del hist
    K.clear_session()
    gc.collect()
    return res

print('Fonctions chargées.')

Fonctions chargées.


## 2. Chargement des données (scale=1.0)

In [9]:
def _compute_lesion_bin(mask_path):
    mask = tf.image.decode_png(tf.io.read_file(mask_path), channels=1)
    ratio = float(tf.reduce_mean(tf.cast(mask > 127, tf.float32)))
    if ratio < 0.01:   return 0, ratio
    elif ratio < 0.05: return 1, ratio
    elif ratio < 0.10: return 2, ratio
    else:              return 3, ratio

def _merge_rare_bins(bins, min_count=4):
    bins = bins.copy()
    unique, counts = np.unique(bins, return_counts=True)
    for b, c in zip(unique, counts):
        if c < min_count:
            bins[bins == b] = b - 1 if b > 0 else b + 1
    return bins

def split_paths(scale=1.0, val_ratio=0.2, seed=SEED):
    imgs  = sorted([os.path.join(IMAGES_DIR, f) for f in os.listdir(IMAGES_DIR) if f.endswith('.jpg')])
    masks = sorted([os.path.join(MASKS_DIR, f) for f in os.listdir(MASKS_DIR) if f.endswith('.png')])
    all_bins = []
    for mp in masks:
        b, _ = _compute_lesion_bin(mp)
        all_bins.append(b)
    all_bins = np.array(all_bins)
    all_bins = _merge_rare_bins(all_bins)
    
    all_idx = np.arange(len(imgs))
    if scale < 1.0:
        sss = StratifiedShuffleSplit(n_splits=1, train_size=scale, random_state=seed)
        keep_idx, _ = next(sss.split(all_idx, all_bins))
        imgs  = [imgs[i] for i in keep_idx]
        masks = [masks[i] for i in keep_idx]
        all_bins = _merge_rare_bins(all_bins[keep_idx])
    
    idx = np.arange(len(imgs))
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=val_ratio, random_state=seed)
    tr_idx, va_idx = next(sss2.split(idx, all_bins))
    to_list = lambda ii: ([imgs[i] for i in ii], [masks[i] for i in ii])
    return to_list(tr_idx), to_list(va_idx)

(train_img, train_mask), (val_img, val_mask) = split_paths(scale=1.0)
print(f'Train: {len(train_img)} | Val: {len(val_img)}')

train_ds = make_ds(train_img, train_mask, augment_fn=augment_flip)
val_ds   = make_ds(val_img, val_mask, shuffle=False)

I0000 00:00:1773518142.634762  336387 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13366 MB memory:  -> device: 0, name: NVIDIA A2, pci bus id: 0000:17:00.0, compute capability: 8.6


Train: 720 | Val: 180


## 3. Entraînement : avec skip vs sans skip

Config identique au modèle FINAL (vérifié dans hyperparam_unet_v2.ipynb) :
`f64/d4/Dice loss/lr=1e-4/flip/batchnorm=True/pooling=max/dropout=0`.
Seule différence : `use_skip`.

**IMPORTANT** : Supprimer `models/skip_full_with.json`, `models/skip_full_with.keras`,
`models/skip_full_without.json`, `models/skip_full_without.keras` avant de relancer
si un run précédent a échoué.

In [10]:
# ── Avec skip connections ──
model_skip = build_unet(use_skip=True)
res_skip = run_experiment('skip_full_with', model_skip, train_ds, val_ds)


  skip_full_with  |  params: 31,402,497
Epoch 1/30


2026-03-14 19:56:01.840552: I external/local_xla/xla/service/service.cc:163] XLA service 0x7f4f9003b1b0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-03-14 19:56:01.840591: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA A2, Compute Capability 8.6
2026-03-14 19:56:02.053621: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-03-14 19:56:03.376612: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 90501
2026-03-14 19:56:24.330315: E external/local_xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng0{} for conv (f32[8,192,256,256]{3,2,1,0}, u8[0]{0}) custom-call(f32[8,64,256,256]{3,2,1,0}, f32[64,192,3,3]{3,2,1,0}), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBackwardInput", backend_config={"operation_qu

90/90 - 154s - 2s/step - binary_accuracy: 0.8003 - dice_coefficient: 0.6422 - iou_coefficient: 0.4809 - loss: 0.3578 - val_binary_accuracy: 0.8235 - val_dice_coefficient: 0.5011 - val_iou_coefficient: 0.3373 - val_loss: 0.4976
Epoch 2/30
90/90 - 67s - 743ms/step - binary_accuracy: 0.8807 - dice_coefficient: 0.7236 - iou_coefficient: 0.5733 - loss: 0.2764 - val_binary_accuracy: 0.8798 - val_dice_coefficient: 0.6848 - val_iou_coefficient: 0.5252 - val_loss: 0.3152
Epoch 3/30
90/90 - 67s - 744ms/step - binary_accuracy: 0.9093 - dice_coefficient: 0.7687 - iou_coefficient: 0.6291 - loss: 0.2313 - val_binary_accuracy: 0.9038 - val_dice_coefficient: 0.7567 - val_iou_coefficient: 0.6144 - val_loss: 0.2430
Epoch 4/30
90/90 - 67s - 743ms/step - binary_accuracy: 0.9108 - dice_coefficient: 0.7786 - iou_coefficient: 0.6427 - loss: 0.2214 - val_binary_accuracy: 0.8821 - val_dice_coefficient: 0.6915 - val_iou_coefficient: 0.5349 - val_loss: 0.3071
Epoch 5/30
90/90 - 67s - 743ms/step - binary_accuracy

In [11]:
# ── Sans skip connections ──
model_noskip = build_unet(use_skip=False)
res_noskip = run_experiment('skip_full_without', model_noskip, train_ds, val_ds)


  skip_full_without  |  params: 28,269,057
Epoch 1/30


2026-03-14 20:31:09.081022: E external/local_xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng0{} for conv (f32[8,128,256,256]{3,2,1,0}, u8[0]{0}) custom-call(f32[8,64,256,256]{3,2,1,0}, f32[64,128,3,3]{3,2,1,0}), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBackwardInput", backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"activation_mode":"kNone","conv_result_scale":1,"side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false,"reification_cost":[]} is taking a while...
2026-03-14 20:31:09.625451: E external/local_xla/xla/service/slow_operation_alarm.cc:140] The operation took 1.544605979s
Trying algorithm eng0{} for conv (f32[8,128,256,256]{3,2,1,0}, u8[0]{0}) custom-call(f32[8,64,256,256]{3,2,1,0}, f32[64,128,3,3]{3,2,1,0}), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBackwardInput", backend_config={"operation_qu

90/90 - 105s - 1s/step - binary_accuracy: 0.8553 - dice_coefficient: 0.7007 - iou_coefficient: 0.5482 - loss: 0.2993 - val_binary_accuracy: 0.8160 - val_dice_coefficient: 0.4830 - val_iou_coefficient: 0.3246 - val_loss: 0.5137
Epoch 2/30
90/90 - 58s - 643ms/step - binary_accuracy: 0.8991 - dice_coefficient: 0.7788 - iou_coefficient: 0.6426 - loss: 0.2212 - val_binary_accuracy: 0.8536 - val_dice_coefficient: 0.6754 - val_iou_coefficient: 0.5149 - val_loss: 0.3251
Epoch 3/30
90/90 - 58s - 644ms/step - binary_accuracy: 0.9102 - dice_coefficient: 0.8035 - iou_coefficient: 0.6751 - loss: 0.1965 - val_binary_accuracy: 0.9032 - val_dice_coefficient: 0.7782 - val_iou_coefficient: 0.6410 - val_loss: 0.2213
Epoch 4/30
90/90 - 58s - 644ms/step - binary_accuracy: 0.9212 - dice_coefficient: 0.8228 - iou_coefficient: 0.7042 - loss: 0.1772 - val_binary_accuracy: 0.8852 - val_dice_coefficient: 0.7749 - val_iou_coefficient: 0.6370 - val_loss: 0.2253
Epoch 5/30
90/90 - 58s - 646ms/step - binary_accuracy

In [12]:
print('\n' + '=' * 60)
print('VALIDATION — skip vs no-skip (scale=1.0)')
print('=' * 60)
for r in results:
    print(f'  {r["name"]:<25} DSC={r["best_val_dice"]:.4f}  IoU={r["best_val_iou"]:.4f}  '
          f'params={r["params"]:,}  epochs={r["stopped_epoch"]}')

d1 = results[0]['best_val_dice']
d2 = results[1]['best_val_dice']
print(f'\nΔ DSC (avec - sans skip) = {d1 - d2:+.4f}')


VALIDATION — skip vs no-skip (scale=1.0)
  skip_full_with            DSC=0.8916  IoU=0.8083  params=31,402,497  epochs=30
  skip_full_without         DSC=0.8948  IoU=0.8111  params=28,269,057  epochs=30

Δ DSC (avec - sans skip) = -0.0032


## 4. Évaluation sur le test set ISBI 2016 (379 images)

In [13]:
# ── Fonctions d'évaluation ──
def dice_np(y_true, y_pred):
    inter = np.sum(y_true * y_pred)
    denom = np.sum(y_true) + np.sum(y_pred)
    return 2.0 * inter / denom if denom > 0 else 1.0

def iou_np(y_true, y_pred):
    inter = np.sum(y_true * y_pred)
    union = np.sum(y_true) + np.sum(y_pred) - inter
    return inter / union if union > 0 else 1.0

def evaluate_model_on_test(model, img_paths, mask_paths):
    per_image = []
    for ip, mp in zip(img_paths, mask_paths):
        img = np.array(Image.open(ip).convert('RGB').resize(
            (IMG_SIZE, IMG_SIZE), Image.BILINEAR)).astype(np.float32) / 255.0
        gt = np.array(Image.open(mp).convert('L').resize(
            (IMG_SIZE, IMG_SIZE), Image.NEAREST))
        gt = (gt > 127).astype(np.float32)
        
        pred_prob = model.predict(img[np.newaxis, ...], verbose=0)[0, :, :, 0]
        pred_bin = (pred_prob >= 0.5).astype(np.float32)
        
        rho = float(np.mean(gt))
        per_image.append({
            'image': Path(ip).stem,
            'dice': float(dice_np(gt, pred_bin)),
            'iou': float(iou_np(gt, pred_bin)),
            'rho': rho,
        })
    return per_image

# Chemins test set
test_img_paths  = sorted([os.path.join(TEST_IMG_DIR, f) for f in os.listdir(TEST_IMG_DIR) if f.endswith('.jpg')])
test_mask_paths = sorted([os.path.join(TEST_MASK_DIR, f) for f in os.listdir(TEST_MASK_DIR) if f.endswith('.png')])
print(f'Test set : {len(test_img_paths)} images')

Test set : 379 images


In [14]:
# ── Évaluer les deux modèles sur le test set ──
test_results = {}

for name in ['skip_full_with', 'skip_full_without']:
    model_path = os.path.join(SAVE_DIR, f'{_safe_name(name)}.keras')
    print(f'\nChargement {model_path} ...')
    model = tf.keras.models.load_model(model_path, custom_objects=CUSTOM_OBJECTS)
    
    print(f'Évaluation sur {len(test_img_paths)} images de test ...')
    per_image = evaluate_model_on_test(model, test_img_paths, test_mask_paths)
    
    dsc_arr = np.array([r['dice'] for r in per_image])
    iou_arr = np.array([r['iou'] for r in per_image])
    
    test_results[name] = {
        'per_image': per_image,
        'dice_mean': float(dsc_arr.mean()),
        'dice_std': float(dsc_arr.std()),
        'iou_mean': float(iou_arr.mean()),
        'iou_std': float(iou_arr.std()),
        'failures': int(np.sum(dsc_arr < 0.5)),
    }
    
    print(f'  DSC = {dsc_arr.mean():.4f} ± {dsc_arr.std():.4f}')
    print(f'  IoU = {iou_arr.mean():.4f} ± {iou_arr.std():.4f}')
    print(f'  Échecs (DSC<0.5) = {np.sum(dsc_arr < 0.5)}')
    
    del model
    K.clear_session()
    gc.collect()


Chargement models/skip_full_with.keras ...
Évaluation sur 379 images de test ...
  DSC = 0.8915 ± 0.1207
  IoU = 0.8204 ± 0.1508
  Échecs (DSC<0.5) = 7

Chargement models/skip_full_without.keras ...
Évaluation sur 379 images de test ...
  DSC = 0.8772 ± 0.1341
  IoU = 0.7998 ± 0.1588
  Échecs (DSC<0.5) = 11


In [15]:
# ── Comparaison finale ──
print('\n' + '=' * 70)
print('COMPARAISON SKIP vs NO-SKIP — DONNÉES COMPLÈTES + TEST SET')
print('=' * 70)

print(f'\n{"Config":<20} {"Val DSC":>10} {"Test DSC":>18} {"Test IoU":>18} {"Échecs":>8}')
print('-' * 70)
for name, label in [('skip_full_with', 'Avec skip'), ('skip_full_without', 'Sans skip')]:
    r = [x for x in results if x['name'] == name][0]
    t = test_results[name]
    print(f'{label:<20} {r["best_val_dice"]:>10.4f} '
          f'{t["dice_mean"]:>8.4f} ± {t["dice_std"]:.4f} '
          f'{t["iou_mean"]:>8.4f} ± {t["iou_std"]:.4f} '
          f'{t["failures"]:>8}')

delta_test = test_results['skip_full_with']['dice_mean'] - test_results['skip_full_without']['dice_mean']
print(f'\nΔ DSC test (avec - sans skip) = {delta_test:+.4f}')

if delta_test > 0.005:
    print('→ Les skip connections AMÉLIORENT les performances sur données complètes.')
    print('  Le résultat de l\'ablation sur 50% était un artefact du sous-échantillon.')
elif delta_test < -0.005:
    print('→ Les skip connections DÉGRADENT les performances, même sur données complètes.')
    print('  Le résultat de l\'ablation est confirmé.')
else:
    print('→ Pas de différence significative. L\'effet des skip connections est marginal.')


COMPARAISON SKIP vs NO-SKIP — DONNÉES COMPLÈTES + TEST SET

Config                  Val DSC           Test DSC           Test IoU   Échecs
----------------------------------------------------------------------
Avec skip                0.8916   0.8915 ± 0.1207   0.8204 ± 0.1508        7
Sans skip                0.8948   0.8772 ± 0.1341   0.7998 ± 0.1588       11

Δ DSC test (avec - sans skip) = +0.0143
→ Les skip connections AMÉLIORENT les performances sur données complètes.
  Le résultat de l'ablation sur 50% était un artefact du sous-échantillon.


In [16]:
# ── Sauvegarde ──
output = {
    'comparison': {
        name: {
            'val_dice': [x for x in results if x['name'] == name][0]['best_val_dice'],
            'val_iou': [x for x in results if x['name'] == name][0]['best_val_iou'],
            'test_dice_mean': test_results[name]['dice_mean'],
            'test_dice_std': test_results[name]['dice_std'],
            'test_iou_mean': test_results[name]['iou_mean'],
            'test_iou_std': test_results[name]['iou_std'],
            'test_failures': test_results[name]['failures'],
        }
        for name in ['skip_full_with', 'skip_full_without']
    },
    'delta_test_dice': delta_test,
}

with open('results_skip_ablation.json', 'w') as f:
    _json.dump(output, f, indent=2)
print('Sauvegardé : results_skip_ablation.json')

Sauvegardé : results_skip_ablation.json
